In [1]:
import anndata as ad
import numpy as np
import pandas as pd
import SDP_miRNA.dataset
import SDP_miRNA.optimization
import SDP_miRNA.optimization_MOSEK
import SDP_miRNA.correlation
import SDP_miRNA.simulation
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from itertools import combinations

In [2]:
rng = np.random.default_rng(392)

# (1) Simulated data: MRR model

Pair of conditionally independent mRNA, with indirect positive correlation induced by miRNA regulation

miRNA + mRNA_1 -> 0 \
miRNA + mRNA_2 -> 0

### Simulation

In [3]:
k_tx = 2
k_deg = 1
k_reg = 2

targets = 2

stoch_inp = np.zeros((2 + targets * 3, 1 + targets))
stoch_out = np.zeros((2 + targets * 3, 1 + targets))
rates = np.ones(2 + 3 * targets)

# miRNA tx and deg reactions
stoch_out[0, 0] = 1
rates[0] = k_tx
stoch_inp[1, 0] = 1
rates[1] = k_deg

for m in range(targets):

    r = 2 + 3 * m
    s = 1 + m

    # mRNA tx and deg reactions
    stoch_out[r, s] = 1
    rates[r] = k_tx
    stoch_inp[r + 1, s] = 1
    rates[r + 1] = k_deg

    # miRNA - mRNA interaction
    stoch_inp[r + 2, 0] = 1
    stoch_inp[r + 2, s] = 1
    stoch_out[r + 2, 0] = 1
    rates[r + 2] = k_reg

In [4]:
initial = None
tmax = 10100
tmin = 100
tint = 10
n = 1000

# run gillespie
path, path_times = SDP_miRNA.simulation.gillespie(stoch_inp, stoch_out, rates, initial, tmax)

In [5]:
# take samples at uniform time points
sample = SDP_miRNA.simulation.uniform_time_samples(path, path_times, tmin=tmin, tmax=tmax, tint=tint, n=n)

### (0) Correlation

In [23]:
Sigma = np.corrcoef(sample.T)
Sigma

array([[ 1.        , -0.30271616, -0.32540215],
       [-0.30271616,  1.        ,  0.2210498 ],
       [-0.32540215,  0.2210498 ,  1.        ]])

- negative correlation miRNA - mRNA
- positive correlation mRNA - mRNA

### (1) Gaussian assumption

In [14]:
Cov = np.cov(sample.T)
Cov_cond = Cov[1:, 1:] - Cov[1:, :1] @ np.linalg.inv(Cov[:1, :1]) @ Cov[:1, 1:]
Cov_cond

array([[0.58978668, 0.08257191],
       [0.08257191, 0.62521352]])

In [16]:
D = np.sqrt(np.diagonal(Cov_cond))
Sigma_cond = Cov_cond / np.outer(D, D)
Sigma_cond

array([[1.        , 0.13597863],
       [0.13597863, 1.        ]])

- mRNA - mRNA | miRNA correlation is weaker than unconditional correlation, but not 0

### (2) Partial correlation

In [24]:
# X, Y = mRNA, Z = miRNA
rho_XY = Sigma[1, 2]
rho_XZ = Sigma[1, 0]
rho_YZ = Sigma[2, 0]

rho_partial = (rho_XY - rho_XZ*rho_YZ) / np.sqrt((1 - rho_XZ**2) * (1 - rho_YZ**2))
rho_partial

np.float64(0.1359786313936149)

- actually equal to the value produced by gaussian assumption

### (3) Precision matrix

In [25]:
Omega = np.linalg.inv(Sigma)
Omega

array([[ 1.19315373,  0.28950959,  0.32425875],
       [ 0.28950959,  1.12162054, -0.15372695],
       [ 0.32425875, -0.15372695,  1.13949581]])

In [26]:
- Omega[1, 2] / np.sqrt(Omega[1, 1] * Omega[2, 2])

np.float64(0.1359786313936149)

In [28]:
D = np.sqrt(np.diagonal(Omega))
-Omega / np.outer(D, D)

array([[-1.        , -0.25026019, -0.27809113],
       [-0.25026019, -1.        ,  0.13597863],
       [-0.27809113,  0.13597863, -1.        ]])

- hard to interpret matrix
- can compute partial correlations

### (4) Conditional Mutual Information

In [ ]:
def entropy_discrete(data):
    """
    Computes Shannon entropy H(V) in bits for a discrete matrix or vector.
    Each unique row is treated as a unique joint state.
    """
    # Convert to a DataFrame to easily group and count unique joint states
    df = pd.DataFrame(data)
    
    # Calculate empirical probabilities of each unique state
    counts = df.value_counts(normalize=True).values
    
    # Standard Shannon entropy formula: -sum(p * log2(p))
    return -np.sum(counts * np.log2(counts))

def estimate_cmi_discrete(X, Y, Z):
    """
    Calculates exact Conditional Mutual Information I(X; Y | Z) for discrete data.
    Returns the metric in 'bits'.
    
    Parameters:
        X, Y, Z : 1D or 2D arrays/lists of discrete/categorical data.
    """
    # Ensure inputs are 2D arrays for clean horizontal stacking
    X = np.atleast_2d(X).T if X.ndim == 1 else X
    Y = np.atleast_2d(Y).T if Y.ndim == 1 else Y
    Z = np.atleast_2d(Z).T if Z.ndim == 1 else Z
    
    # Construct the joint distributions
    XZ = np.hstack((X, Z))
    YZ = np.hstack((Y, Z))
    XYZ = np.hstack((X, Y, Z))
    
    # Compute individual joint entropies
    H_XZ = entropy_discrete(XZ)
    H_YZ = entropy_discrete(YZ)
    H_XYZ = entropy_discrete(XYZ)
    H_Z = entropy_discrete(Z)
    
    # Apply the CMI formula
    cmi = H_XZ + H_YZ - H_XYZ - H_Z
    
    # Clip tiny negative numbers that occur due to floating-point rounding errors
    return max(0.0, cmi)

In [40]:
# Example 1: Conditional Independence (Z determines both X and Y)
# If we know Z, knowing X gives us absolutely zero new information about Y.
Z_ind = np.array([0, 0, 0, 0, 1, 1, 1, 1])
X_ind = np.array([1, 1, 0, 0, 1, 1, 0, 0])
Y_ind = np.array([0, 0, 0, 0, 1, 1, 1, 1]) # Exactly matches Z's patterns, independent of X

cmi_ind = estimate_cmi_discrete(X_ind, Y_ind, Z_ind)
print(f"Scenario 1 (Conditionally Independent): I(X;Y|Z) = {cmi_ind:.4f} bits")

# Example 2: The XOR Spreading / Collider Effect (X and Y are dependent given Z)
# X and Y look completely random and independent on their own.
# However, if you know the outcome Z (where Z = X XOR Y), knowing X perfectly reveals Y.
X_xor = np.array([0, 0, 1, 1, 0, 0, 1, 1])
Y_xor = np.array([0, 1, 0, 1, 0, 1, 0, 1])
Z_xor = np.bitwise_xor(X_xor, Y_xor) # Z is the XOR result

cmi_dep = estimate_cmi_discrete(X_xor, Y_xor, Z_xor)
print(f"Scenario 2 (XOR / Conditionally Dependent): I(X;Y|Z) = {cmi_dep:.4f} bits")

Scenario 1 (Conditionally Independent): I(X;Y|Z) = 0.0000 bits
Scenario 2 (XOR / Conditionally Dependent): I(X;Y|Z) = 1.0000 bits


In [ ]:
# mRNA, mRNA | miRNA
cmi = estimate_cmi_discrete(sample[:, 1], sample[:, 2], sample[:, 0])
cmi

np.float64(0.042792436101820375)

In [43]:
# mRNA, miRNA | mRNA
cmi_1 = estimate_cmi_discrete(sample[:, 0], sample[:, 1], sample[:, 2])
cmi_2 = estimate_cmi_discrete(sample[:, 0], sample[:, 2], sample[:, 1])
cmi_1, cmi_2

(np.float64(0.09708899283993921), np.float64(0.11359210591577606))

In [46]:
estimate_cmi_discrete(sample[:, 0], sample[:, 0], sample[:, 2])

np.float64(2.3568372505305373)